# TP1 — Baselines — Fase 5

**73.69 Large Language Models — ITBA, 2026** · Predicción de *Buy Through Rate*

Este notebook es el entregable de la **Fase 5** del [plan](../plan.md): los baselines contra
los que se va a comparar el Transformer, corridos **antes** de escribir una sola línea del
modelo.

Sirven para dos cosas distintas:

1. **Validar el pipeline.** Si el baseline solo-tabular no reproduce ROC ≈ 0.58 y el de
   TF-IDF sobre el título no reproduce ≈ 0.956, el bug está en la Fase 4 y no en un modelo
   que todavía no existe.
2. **Fijar el objetivo.** El GBDT con el sufijo explícito es el baseline fuerte. Si el
   Transformer da mucho más, hay fuga; si da mucho menos, hay un bug.

## Protocolo

| | |
|---|---|
| Partición | **CV agrupada de 5 folds sobre `dev`**, congelada en la Fase 3 |
| Agrupamiento | `query_id`: las filas de una búsqueda van siempre juntas |
| Test | **no se toca**. Se evalúa una sola vez, en la Fase 10 |
| Ajuste | `FeaturePipeline` y `TfidfVectorizer` se ajustan **por fold, solo con train** |
| Métrica principal | PR-AUC (`average_precision_score`), reportada junto a la prevalencia |
| Métricas secundarias | ROC-AUC, log loss, Brier |

## Índice

| § | Contenido |
|---|---|
| 1 | Qué es cada baseline |
| 2 | Resultados: media ± desvío sobre los 5 folds |
| 3 | Verificación contra los valores esperados del plan |
| 4 | Curvas ROC y PR sobre las predicciones *out-of-fold* |
| 5 | Lecturas y consecuencias para el Transformer |

---
## Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# El notebook corre desde notebooks/; la raiz del repo es el padre.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_curve

from src import baselines as B
from src.data.load import TARGET, load_raw
from src.data.splits import load_splits
from src.evaluate import RESULTS_CSV, compute_metrics, format_summary, load_results, summarize

FIGDIR = ROOT / "report" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 190)
pd.set_option("display.max_colwidth", 95)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "font.size": 9,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

AZUL, GRIS, ROJO, VERDE = "#2b6cb0", "#a0aec0", "#c53030", "#2f855a"


def guardar(fig, nombre):
    """Guarda la figura en report/figures/ para llevarla a la presentacion."""
    ruta = FIGDIR / f"{nombre}.png"
    fig.savefig(ruta)
    print(f"guardada: {ruta.relative_to(ROOT)}")


df = load_raw()
splits = load_splits()
y = df[TARGET].astype(int).to_numpy()
print(f"dev: {len(splits.dev):,} filas en {splits.n_folds} folds  |  "
      f"test: {len(splits.test):,} filas, intacto")

---
## 1. Qué es cada baseline

In [ ]:
catalogo = pd.DataFrame(
    [{"baseline": b.nombre, "features": b.features, "que muestra": b.notas} for b in B.BASELINES]
)
catalogo

Los seis que pide el plan son `prior`, `logreg_tabular`, `gbdt_tabular`, `tfidf_*`,
`gbdt_sufijo` y `mlp_texto_tabular`. El de TF-IDF se corre en cuatro variantes porque las
cuatro contestan preguntas distintas y cuestan lo mismo: **solo título** (es el número que
el plan usa como control del pipeline), **solo descripción** y **ambos** anticipan la
ablación de campo de texto de la Fase 9, y **sin marcador** anticipa la ablación central.

---
## 2. Resultados

In [ ]:
# Los resultados viven en experiments/results.csv, una fila por baseline y fold.
# Poner RECOMPUTAR = True para volver a correrlos (~4 minutos, casi todo el MLP).
RECOMPUTAR = False

resultados = load_results()
baselines_cv = resultados.loc[
    (resultados["config"] == "baseline") & (resultados["split"] == "cv_dev")
]

if RECOMPUTAR or baselines_cv.empty:
    from src.evaluate import append_results
    filas = B.run_cv(verbose=True)
    append_results(filas)
    resultados = load_results()
    baselines_cv = resultados.loc[
        (resultados["config"] == "baseline") & (resultados["split"] == "cv_dev")
    ]

print(f"{len(baselines_cv)} corridas leidas de {RESULTS_CSV.relative_to(ROOT)} "
      f"({baselines_cv['run_name'].nunique()} baselines x {baselines_cv['fold'].nunique()} folds)")

In [ ]:
resumen = summarize(baselines_cv)
tabla = format_summary(resumen)
tabla.insert(1, "features", resumen["features"])
tabla

El `lift` es `PR-AUC / prevalencia`: cuántas veces mejor que predecir la tasa base. Es la
forma honesta de leer un PR-AUC, porque su piso no es 0 sino la prevalencia.

In [ ]:
# Variabilidad entre folds: es lo que define si una diferencia es real o ruido.
por_fold = baselines_cv.pivot(index="run_name", columns="fold", values="pr_auc")
por_fold = por_fold.reindex([b.nombre for b in B.BASELINES])
por_fold["media"] = por_fold.mean(axis=1)
por_fold["desvio"] = por_fold.iloc[:, :5].std(axis=1, ddof=1)
por_fold["rango"] = por_fold.iloc[:, :5].max(axis=1) - por_fold.iloc[:, :5].min(axis=1)
por_fold

El desvío entre folds del PR-AUC es de 3 a 5 puntos en los baselines de texto. Con ~200
positivos por fold de validación eso es lo esperable, y es la razón por la que la Fase 9
corre **5 semillas × 5 folds**: sin desvío, una tabla de ablación no permite concluir nada.

In [ ]:
orden = [b.nombre for b in B.BASELINES]
res = resumen.set_index("run_name").reindex(orden)
prevalencia = float(baselines_cv["prevalence"].mean())

# Un color por familia, igual en los dos paneles: gris = no ve el marcador de
# reputacion, azul = lo ve por el texto, rojo = lo recibe servido como categorica.
SIN_SENAL = {"prior", "logreg_tabular", "gbdt_tabular", "tfidf_sin_marcador"}
COLOR_FAMILIA = {n: (ROJO if n == "gbdt_sufijo" else GRIS if n in SIN_SENAL else AZUL)
                 for n in orden}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (metrica, piso, etiqueta) in zip(
    axes,
    [("roc_auc", 0.5, "ROC-AUC"), ("pr_auc", prevalencia, "PR-AUC (average precision)")],
):
    colores = [COLOR_FAMILIA[n] for n in orden]
    ax.barh(orden, res[metrica], xerr=res[f"{metrica}_sd"], color=colores,
            edgecolor="white", capsize=3)
    ax.axvline(piso, color="black", ls="--", lw=1)
    ax.text(piso, -0.8, f"  piso = {piso:.4f}", fontsize=8, va="top")
    ax.set_xlabel(etiqueta)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.02)
axes[0].set_title("ROC-AUC por baseline (media ± desvío sobre 5 folds)", loc="left")
axes[1].set_title("PR-AUC: el piso es la prevalencia, no cero", loc="left")
fig.tight_layout()
guardar(fig, "fig_09_baselines")

---
## 3. Verificación contra los valores esperados del plan

El plan (sección 0.6) declara tres números medidos sobre un único `GroupShuffleSplit`. Acá
se comparan contra la media de la CV agrupada, que es una partición distinta: se espera
coincidencia aproximada, no exacta. El criterio de aceptación de la fase son las dos
primeras filas.

In [ ]:
esperado = [
    ("gbdt_tabular", "roc_auc", 0.5789, 0.05, "criterio de aceptacion: solo-tabular ~ 0.58"),
    ("tfidf_title", "roc_auc", 0.9562, 0.02, "criterio de aceptacion: TF-IDF de titulo ~ 0.956"),
    ("gbdt_sufijo", "roc_auc", 0.9695, 0.02, "baseline fuerte de referencia"),
    ("gbdt_sufijo", "pr_auc", 0.7750, 0.06, "objetivo realista del Transformer"),
    ("prior", "pr_auc", prevalencia, 0.01, "el prior tiene que dar la prevalencia"),
]
check = pd.DataFrame([
    {
        "baseline": n, "metrica": m, "plan": v, "medido": res[m][n],
        "delta": abs(res[m][n] - v), "tolerancia": tol, "ok": abs(res[m][n] - v) <= tol,
        "que verifica": nota,
    }
    for n, m, v, tol, nota in esperado
])
assert check["ok"].all(), f"no se reproducen:\n{check.loc[~check['ok']].to_string(index=False)}"
check

---
## 4. Curvas ROC y PR sobre predicciones *out-of-fold*

Las curvas se arman con la predicción que cada fila de `dev` recibió **del fold en el que
fue validación**, así que ninguna fila está evaluada por un modelo que la vio entrenando.
Se recalculan acá porque `results.csv` guarda métricas, no predicciones. Se dejan afuera
las variantes redundantes y el MLP, que es el único caro.

In [ ]:
def predicciones_oof(baseline: B.Baseline, seed: int = B.SEED) -> np.ndarray:
    """Predice cada fila de dev con el modelo del fold donde fue validacion."""
    oof = np.full(len(df), np.nan)
    for tr_idx, ev_idx in splits.folds:
        prob, _ = baseline.fn(df.iloc[tr_idx], df.iloc[ev_idx], seed)
        oof[ev_idx] = prob
    return oof[splits.dev]


elegidos = ["prior", "gbdt_tabular", "tfidf_sin_marcador", "tfidf_texto", "gbdt_sufijo"]
oof = {n: predicciones_oof(next(b for b in B.BASELINES if b.nombre == n)) for n in elegidos}
y_dev = y[splits.dev]

pd.DataFrame([
    {"baseline": n, **{k: round(v, 4) for k, v in compute_metrics(y_dev, p).items()}}
    for n, p in oof.items()
])

Las métricas *out-of-fold* agregadas no son idénticas a la media de las métricas por fold
—se calculan sobre el pool completo, con una sola prevalencia— pero deben quedar muy cerca.
Que lo hagan es una verificación más de que el ensamblado de las predicciones está bien.

La única excepción es el `prior`: por fold predice una constante y su ROC-AUC vale 0.5 por
convención, pero al juntar los cinco folds las cinco constantes son levemente distintas
(la prevalencia de cada train), así que el pool sí tiene un orden — y es un orden que no
informa nada, de ahí el 0.487.

In [ ]:
COLOR = {"prior": GRIS, "gbdt_tabular": "#805ad5", "tfidf_sin_marcador": "#dd6b20",
         "tfidf_texto": AZUL, "gbdt_sufijo": ROJO}

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 4.8))
for nombre, prob in oof.items():
    m = compute_metrics(y_dev, prob)
    fpr, tpr, _ = roc_curve(y_dev, prob)
    ax_roc.plot(fpr, tpr, color=COLOR[nombre], lw=1.6,
                label=f"{nombre}  (ROC {m['roc_auc']:.3f})")
    precision, recall, _ = precision_recall_curve(y_dev, prob)
    ax_pr.plot(recall, precision, color=COLOR[nombre], lw=1.6,
               label=f"{nombre}  (PR {m['pr_auc']:.3f})")

ax_roc.plot([0, 1], [0, 1], color="black", ls="--", lw=1)
ax_roc.set(xlabel="tasa de falsos positivos", ylabel="tasa de verdaderos positivos")
ax_roc.set_title("ROC — out-of-fold sobre dev", loc="left")
ax_roc.legend(loc="lower right", fontsize=8)

prev_dev = float(y_dev.mean())
ax_pr.axhline(prev_dev, color="black", ls="--", lw=1)
ax_pr.text(0.02, prev_dev + 0.02, f"prevalencia = {prev_dev:.4f}", fontsize=8)
ax_pr.set(xlabel="recall", ylabel="precision", ylim=(0, 1.02))
ax_pr.set_title("Precision-Recall — el piso es la prevalencia", loc="left")
ax_pr.legend(loc="upper right", fontsize=8)
fig.tight_layout()
guardar(fig, "fig_10_curvas_baselines")

---
## 5. Lecturas

In [ ]:
lecturas = pd.DataFrame([
    {
        "hallazgo": "El texto es la señal, y lo tabular casi no aporta solo",
        "evidencia": f"gbdt_tabular ROC {res['roc_auc']['gbdt_tabular']:.4f} contra "
                     f"tfidf_texto {res['roc_auc']['tfidf_texto']:.4f}",
    },
    {
        "hallazgo": "Sin el marcador de reputación el problema es irresoluble",
        "evidencia": f"tfidf_sin_marcador ROC {res['roc_auc']['tfidf_sin_marcador']:.4f}, "
                     f"PR {res['pr_auc']['tfidf_sin_marcador']:.4f} contra una prevalencia "
                     f"de {prevalencia:.4f}",
    },
    {
        "hallazgo": "Título y descripción son redundantes entre sí",
        "evidencia": f"solo título {res['roc_auc']['tfidf_title']:.4f}, solo descripción "
                     f"{res['roc_auc']['tfidf_description']:.4f}, ambos "
                     f"{res['roc_auc']['tfidf_texto']:.4f}: la diferencia cabe en un desvío",
    },
    {
        "hallazgo": "La fusión texto + tabular es donde está el margen",
        "evidencia": f"gbdt_sufijo PR {res['pr_auc']['gbdt_sufijo']:.4f} contra tfidf_texto "
                     f"{res['pr_auc']['tfidf_texto']:.4f}: +"
                     f"{100 * (res['pr_auc']['gbdt_sufijo'] - res['pr_auc']['tfidf_texto']):.1f} "
                     f"puntos que aportan allergens y category dentro del nivel ALTO",
    },
    {
        "hallazgo": "Una red neuronal sin atención ya llega cerca",
        "evidencia": f"mlp_texto_tabular PR {res['pr_auc']['mlp_texto_tabular']:.4f} "
                     f"contra gbdt_sufijo {res['pr_auc']['gbdt_sufijo']:.4f}",
    },
])
lecturas

### Consecuencias para el Transformer

1. **El objetivo realista es PR-AUC ≈ 0.78–0.81 y ROC ≈ 0.97**, que es lo que da el GBDT con
   el sufijo servido como categórica. Superarlo por mucho sería señal de fuga, no de mérito.
2. **Es previsible que el GBDT gane.** La señal es un token categórico de 20 valores: no
   requiere composicionalidad, que es justamente lo que un encoder aporta. Ese resultado,
   presentado con el análisis de por qué, vale más que un número inflado.
3. **El margen del Transformer está en la fusión**, no en el texto solo: `tfidf_texto`
   (0.685) contra `gbdt_sufijo` (0.808) son 12 puntos de PR-AUC que vienen de `allergens` y
   `category` modulando la compra dentro del nivel ALTO.
4. **El `mlp_texto_tabular` es el control de arquitectura.** Ya combina texto y tabular sin
   atención. Si el Transformer no le gana, lo que aporte no será el mecanismo de atención.
5. **`tfidf_sin_marcador` (ROC 0.524) es la ablación central adelantada.** El modelo no
   aprende "qué productos se compran": aprende a leer un marcador de reputación insertado
   sintéticamente. Conviene decirlo en la presentación antes de que lo pregunten.